# Reconhecimento de Entidades Nomeadas (NER) com spaCy

**Objetivo**: extrair pessoas, organizações e lugares de texto em português, **medir** a qualidade
dessa extração contra um corpus anotado e estender o modelo com regras próprias.

**Modelo**: `pt_core_news_sm` — pipeline português do spaCy, treinado em WikiNER. Reconhece quatro
tipos: `PER`, `ORG`, `LOC` e `MISC`.

**Corpus de avaliação**: [WikiNEuRal](https://huggingface.co/datasets/Babelscape/wikineural),
split `test_pt` — 10.160 frases em português anotadas no esquema BIO com os mesmos quatro tipos.
Ter o mesmo conjunto de rótulos é o que permite comparar direto, sem mapeamento.

**Roteiro**

| Seção | Conteúdo |
|---|---|
| 1 | Anatomia de um `Doc`: tokens, spans, rótulos |
| 2 | O corpus anotado e o esquema BIO |
| 3 | Rodar o modelo com tokenização alinhada ao gold |
| 4 | Precisão, revocação e F1 **calculados na mão** |
| 5 | Análise de erros: fronteira, tipo, omissão |
| 6 | `EntityRuler`: entidades que o modelo não conhece |
| 7 | Limitações e exercícios |

> A primeira execução baixa o corpus para `data/` — são ~400 MB, porque o WikiNEuRal é
> publicado com os nove idiomas juntos e usamos só o `test_pt`. O modelo do spaCy já vem
> instalado como dependência do projeto.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "ner" else Path.cwd()
os.environ.setdefault("HF_HOME", str(PROJECT_DIR / "data" / "hf"))

import pandas as pd
import spacy
from spacy import displacy
from spacy.tokens import Doc

pd.set_option("display.max_colwidth", 70)

nlp = spacy.load("pt_core_news_sm")
print("pipeline:", nlp.pipe_names)
print("rótulos de entidade:", nlp.get_pipe("ner").labels)

pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']
rótulos de entidade: ('LOC', 'MISC', 'ORG', 'PER')


## 1. Anatomia de um `Doc`

`nlp(texto)` devolve um `Doc`: uma sequência de tokens com os atributos que cada componente do
pipeline anotou. NER é o último componente, e o resultado aparece em `doc.ents` — uma lista de
`Span`, cada um com rótulo, posição em tokens e posição em caracteres.

O ponto a fixar: uma entidade é um **intervalo de tokens**, não uma palavra. `Rio de Janeiro` é um
único `Span` de três tokens.

In [2]:
texto = (
    "Em 2019, Roberto Silva deixou a Petrobras para fundar a Aurora Energia no Rio de Janeiro, "
    "com apoio do BNDES e da Universidade Federal de Minas Gerais."
)
doc = nlp(texto)

pd.DataFrame([
    {"token": t.text, "lema": t.lemma_, "classe": t.pos_,
     "entidade": t.ent_type_ or "-", "posicao_BIO": t.ent_iob_}
    for t in doc
]).head(20)

,token,lema,classe,entidade,posicao_BIO
0,Em,em,ADP,-,O
1,2019,2019,NUM,-,O
2,",",",",PUNCT,-,O
3,Roberto,Roberto,PROPN,PER,B
4,Silva,Silva,PROPN,PER,I
5,deixou,deixar,VERB,-,O
6,a,a,DET,-,O
7,Petrobras,Petrobras,PROPN,ORG,B
8,para,para,SCONJ,-,O
9,fundar,fundar,VERB,-,O


In [3]:
pd.DataFrame([
    {"entidade": ent.text, "rotulo": ent.label_,
     "significado": spacy.explain(ent.label_),
     "tokens": f"{ent.start}:{ent.end}", "caracteres": f"{ent.start_char}:{ent.end_char}"}
    for ent in doc.ents
])

,entidade,rotulo,significado,tokens,caracteres
0,Roberto Silva,PER,Named person or family.,3:5,9:22
1,Petrobras,ORG,"Companies, agencies, institutions, etc.",7:8,32:41
2,Aurora Energia,LOC,"Non-GPE locations, mountain ranges, bodies of water",11:13,56:70
3,Rio de Janeiro,LOC,"Non-GPE locations, mountain ranges, bodies of water",14:17,74:88
4,BNDES,MISC,"Miscellaneous entities, e.g. events, nationalities, products or wo...",21:22,103:108
5,Universidade Federal de Minas Gerais,LOC,"Non-GPE locations, mountain ranges, bodies of water",24:29,114:150


In [4]:
# Visualização: útil para conferir fronteiras de olho.
displacy.render(doc, style="ent", jupyter=True)

### O que o `IOB` da tabela acima significa

Cada token recebe uma marca de posição dentro da entidade:

- `B` — *begin*, primeiro token da entidade
- `I` — *inside*, continuação
- `O` — *outside*, fora de qualquer entidade

Combinando marca e tipo (`B-PER`, `I-PER`, `O`) dá para representar entidades de vários tokens sem
ambiguidade, e por isso NER é tratado como **classificação por token**. É exatamente nesse formato
que o corpus anotado vem.

## 2. O corpus anotado

O WikiNEuRal entrega cada frase já tokenizada (`tokens`) e um inteiro por token (`ner_tags`). O
mapeamento dos inteiros é o esquema BIO padrão de 9 classes.

In [5]:
from datasets import load_dataset

gold = load_dataset("Babelscape/wikineural", split="test_pt")

ID2BIO = {
    0: "O",
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC",
}
ROTULOS = ["PER", "ORG", "LOC", "MISC"]

print("frases:", gold.num_rows)
i = 0
pd.DataFrame({"token": gold["tokens"][i],
              "tag_id": gold["ner_tags"][i],
              "tag": [ID2BIO[t] for t in gold["ner_tags"][i]]}).T

frases: 10160


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
token,Por,volta,de,1874,",",a,sede,do,governo,do,condado,foi,transferida,de,Mineola,para,Long,Island,City,.
tag_id,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,5,6,6,0
tag,O,O,O,O,O,O,O,O,O,O,O,O,O,O,B-LOC,O,B-LOC,I-LOC,I-LOC,O


### De BIO para spans

Para avaliar não basta comparar token por token: queremos saber se a **entidade inteira** foi
encontrada, com o tipo certo e as fronteiras certas. Então convertemos as sequências BIO em
conjuntos de spans `(rótulo, início, fim)`.

Um detalhe de robustez: uma sequência pode começar com `I-PER` sem `B-PER` antes (acontece em
corpus real e na saída de modelos). Tratamos esse `I-` órfão como início de entidade, convenção
usada pelo `seqeval` no modo permissivo.

In [6]:
def bio_para_spans(tags: list[str]) -> set[tuple[str, int, int]]:
    '''Converte uma sequência BIO em {(rótulo, início_token, fim_token_exclusivo)}.'''
    spans, inicio, rotulo_atual = set(), None, None

    def fechar(fim: int) -> None:
        if rotulo_atual is not None:
            spans.add((rotulo_atual, inicio, fim))

    for i, tag in enumerate(tags):
        if tag == "O":
            fechar(i)
            inicio, rotulo_atual = None, None
            continue
        marca, rotulo = tag.split("-", 1)
        if marca == "B" or rotulo != rotulo_atual:  # B-, ou I- que troca de tipo / I- órfão
            fechar(i)
            inicio, rotulo_atual = i, rotulo
    fechar(len(tags))
    return spans

# Conferência rápida em casos construídos à mão.
assert bio_para_spans(["O", "B-PER", "I-PER", "O"]) == {("PER", 1, 3)}
assert bio_para_spans(["B-LOC", "B-LOC"]) == {("LOC", 0, 1), ("LOC", 1, 2)}
assert bio_para_spans(["I-ORG", "I-ORG"]) == {("ORG", 0, 2)}          # I- órfão
assert bio_para_spans(["B-PER", "I-LOC"]) == {("PER", 0, 1), ("LOC", 1, 2)}  # troca de tipo
assert bio_para_spans(["O", "O"]) == set()
print("bio_para_spans ok")

print("\nspans da frase de exemplo:")
tags_exemplo = [ID2BIO[t] for t in gold["ner_tags"][i]]
toks_exemplo = gold["tokens"][i]
for rot, ini, fim in sorted(bio_para_spans(tags_exemplo), key=lambda s: s[1]):
    print(f"  {rot:5s} {' '.join(toks_exemplo[ini:fim])!r}")

bio_para_spans ok

spans da frase de exemplo:
  LOC   'Mineola'
  LOC   'Long Island City'


## 3. Rodando o modelo sobre o gold

Um cuidado indispensável: o spaCy tem o **seu próprio** tokenizador, e ele não quebra o texto
igual ao corpus. Se passássemos a frase remontada com `" ".join(tokens)`, os índices de token do
modelo e do gold não bateriam e a avaliação viraria ruído.

A solução é construir o `Doc` já com a tokenização do corpus (`Doc(nlp.vocab, words=...)`) e rodar
os componentes do pipeline manualmente sobre ele. Assim cada token tem o mesmo índice nos dois
lados.

In [7]:
def prever_pre_tokenizado(tokens: list[str]) -> Doc:
    '''Roda o pipeline sobre uma tokenização já definida, preservando os índices.'''
    doc = Doc(nlp.vocab, words=tokens)
    for _, componente in nlp.pipeline:
        doc = componente(doc)
    return doc

def spans_do_doc(doc: Doc) -> set[tuple[str, int, int]]:
    return {(ent.label_, ent.start, ent.end) for ent in doc.ents}

# Amostra: o corpus inteiro levaria minutos e não muda as conclusões.
N_FRASES = 2_000
amostra = gold.select(range(N_FRASES))

pares = []  # (tokens, spans_gold, spans_previstos)
for tokens, tags in zip(amostra["tokens"], amostra["ner_tags"]):
    doc = prever_pre_tokenizado(tokens)
    pares.append((tokens, bio_para_spans([ID2BIO[t] for t in tags]), spans_do_doc(doc)))

print(f"{len(pares)} frases processadas")
print("entidades no gold:     ", sum(len(g) for _, g, _ in pares))
print("entidades previstas:   ", sum(len(p) for _, _, p in pares))

2000 frases processadas
entidades no gold:      4039
entidades previstas:    4158


In [8]:
# Uma frase com divergência, lado a lado.
def formatar(tokens, spans):
    return ", ".join(f"{r}:{' '.join(tokens[i:f])}" for r, i, f in sorted(spans, key=lambda s: s[1])) or "—"

for tokens, g, p in pares:
    if g != p and g and p:
        print(" ".join(tokens)[:220], "\n")
        print("  gold:     ", formatar(tokens, g))
        print("  previsto: ", formatar(tokens, p))
        break

O município se destacou na década de 1980 e até meados de década de 1990 como maior produtor nacional de algodão , com isso o município que ainda agregava os distridos de Rancho Alegre d' Oeste e Quarto Centenário chegou 

  gold:      LOC:Rancho Alegre d' Oeste, LOC:Quarto Centenário
  previsto:  LOC:Rancho Alegre, LOC:Oeste, LOC:Quarto Centenário


## 4. Precisão, revocação e F1 na mão

A avaliação é feita em **span exato**: o modelo só acerta se o tipo e as duas fronteiras
coincidirem. `LOC:Minas Gerais` quando o gold diz `LOC:Estado de Minas Gerais` conta como erro
duplo — um falso positivo e um falso negativo.

Por rótulo:

$$P = \frac{TP}{TP + FP} \qquad R = \frac{TP}{TP + FN} \qquad F_1 = \frac{2PR}{P + R}$$

- **TP** — span previsto que existe idêntico no gold
- **FP** — span previsto que não existe no gold (invenção ou fronteira/tipo errado)
- **FN** — span do gold que o modelo não produziu

Duas formas de agregar:

- **micro** — soma TP, FP e FN de todos os rótulos antes de dividir. Rótulos frequentes pesam mais.
- **macro** — média aritmética dos F1 por rótulo. Rótulo raro pesa igual a rótulo comum.

In [9]:
def contar(pares) -> dict[str, Counter]:
    '''TP/FP/FN por rótulo, comparando spans exatos.'''
    contagem = defaultdict(Counter)
    for _, g, p in pares:
        for span in p & g:
            contagem[span[0]]["TP"] += 1
        for span in p - g:
            contagem[span[0]]["FP"] += 1
        for span in g - p:
            contagem[span[0]]["FN"] += 1
    return contagem

def prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

contagem = contar(pares)

linhas = []
for rot in ROTULOS:
    c = contagem[rot]
    p, r, f = prf(c["TP"], c["FP"], c["FN"])
    linhas.append({"rotulo": rot, "suporte": c["TP"] + c["FN"], "TP": c["TP"],
                   "FP": c["FP"], "FN": c["FN"],
                   "precisao": round(p, 3), "revocacao": round(r, 3), "f1": round(f, 3)})

tabela = pd.DataFrame(linhas)

tp_tot = sum(contagem[r]["TP"] for r in ROTULOS)
fp_tot = sum(contagem[r]["FP"] for r in ROTULOS)
fn_tot = sum(contagem[r]["FN"] for r in ROTULOS)
p_micro, r_micro, f_micro = prf(tp_tot, fp_tot, fn_tot)

print(f"micro  P={p_micro:.3f}  R={r_micro:.3f}  F1={f_micro:.3f}")
print(f"macro F1 = {tabela['f1'].mean():.3f}")
tabela

micro  P=0.895  R=0.921  F1=0.908
macro F1 = 0.851


,rotulo,suporte,TP,FP,FN,precisao,revocacao,f1
0,PER,740,701,77,39,0.901,0.947,0.924
1,ORG,350,275,53,75,0.838,0.786,0.811
2,LOC,2543,2439,175,104,0.933,0.959,0.946
3,MISC,406,305,133,101,0.696,0.751,0.723


### Conferindo contra uma implementação de referência

O spaCy tem o seu próprio avaliador (`Scorer`), que usa a mesma definição de span exato. Se os
números baterem, nossa contagem está correta.

In [10]:
from spacy.scorer import Scorer
from spacy.tokens import Span
from spacy.training import Example

exemplos = []
for tokens, spans_gold, _ in pares:
    previsto = prever_pre_tokenizado(tokens)
    referencia = Doc(nlp.vocab, words=tokens)
    referencia.ents = [Span(referencia, ini, fim, label=rot) for rot, ini, fim in spans_gold]
    exemplos.append(Example(previsto, referencia))

placar = Scorer().score_spans(exemplos, "ents")
print(f"spaCy Scorer  P={placar['ents_p']:.3f}  R={placar['ents_r']:.3f}  F1={placar['ents_f']:.3f}")
print(f"nossa conta   P={p_micro:.3f}  R={r_micro:.3f}  F1={f_micro:.3f}")

assert abs(placar["ents_f"] - f_micro) < 1e-6, "divergência entre nossa conta e o Scorer"
print("\nMétricas conferem.")

spaCy Scorer  P=0.895  R=0.921  F1=0.908
nossa conta   P=0.895  R=0.921  F1=0.908

Métricas conferem.


## 5. Análise de erros

O F1 agregado não diz *que tipo* de erro o modelo comete. Separamos os falsos positivos em três
categorias, comparando cada um com o gold da mesma frase:

- **fronteira** — o modelo achou a entidade no lugar certo, mas cortou errado
  (`Universidade Federal` em vez de `Universidade Federal de Minas Gerais`)
- **tipo** — fronteiras certas, rótulo errado (`ORG` onde o gold diz `LOC`)
- **inventada** — não há entidade nenhuma naquela região do texto

Essa separação muda o que fazer a seguir: erro de tipo pede mais dados de treino do tipo confundido;
erro de fronteira costuma vir de convenção de anotação diferente.

In [11]:
def classificar_fp(spans_gold, span_fp) -> str:
    rot, ini, fim = span_fp
    for rot_g, ini_g, fim_g in spans_gold:
        if (ini_g, fim_g) == (ini, fim):
            return "tipo"
        if ini < fim_g and ini_g < fim:  # há sobreposição de tokens
            return "fronteira"
    return "inventada"

tipos_erro = Counter()
exemplos_erro = defaultdict(list)
for tokens, g, p in pares:
    for span in p - g:
        categoria = classificar_fp(g, span)
        tipos_erro[categoria] += 1
        if len(exemplos_erro[categoria]) < 3:
            rot, ini, fim = span
            contexto = " ".join(tokens[max(0, ini - 4):fim + 4])
            gold_txt = formatar(tokens, {s for s in g if s[1] < fim and ini < s[2]})
            exemplos_erro[categoria].append(
                {"categoria": categoria, "previsto": f"{rot}:{' '.join(tokens[ini:fim])}",
                 "gold_na_regiao": gold_txt, "contexto": contexto}
            )

total_fp = sum(tipos_erro.values())
for categoria, n in tipos_erro.most_common():
    print(f"{categoria:11s} {n:5d}  ({100 * n / total_fp:.1f}% dos falsos positivos)")

pd.DataFrame([e for lista in exemplos_erro.values() for e in lista])

tipo          174  (39.7% dos falsos positivos)
fronteira     136  (31.1% dos falsos positivos)
inventada     128  (29.2% dos falsos positivos)


,categoria,previsto,gold_na_regiao,contexto
0,fronteira,LOC:Rancho Alegre,LOC:Rancho Alegre d' Oeste,agregava os distridos de Rancho Alegre d' Oeste e Quarto
1,fronteira,LOC:Oeste,LOC:Rancho Alegre d' Oeste,de Rancho Alegre d' Oeste e Quarto Centenário chegou
2,fronteira,LOC:Estádio Antonio Lins,LOC:Estádio Antonio Lins Ribeiro Guimarães,"estádio é o "" Estádio Antonio Lins Ribeiro Guimarães "" com"
3,tipo,PER:Moreira Sales,LOC:Moreira Sales,"municípios de Mariluz , Moreira Sales e Janiópolis os quais"
4,tipo,LOC:Kodokan,MISC:Kodokan,"seu estilo no centro Kodokan e , eventualmente ,"
5,tipo,MISC:Eddie,PER:Eddie,"da torcida é o Eddie , símbolo da banda"
6,inventada,LOC:Ju-Jutsu,—,"Ju-Jutsu , no Japão ,"
7,inventada,PER:nage waza,—,adoptou algumas técnicas de nage waza .
8,inventada,ORG:MVD,—,a informação obtida pelo MVD dos Estados Unidos .


In [12]:
# Falsos negativos: entidades que o modelo não viu. Quais tipos escapam mais?
fn_por_rotulo = Counter()
exemplos_fn = []
for tokens, g, p in pares:
    for span in g - p:
        fn_por_rotulo[span[0]] += 1
        if len(exemplos_fn) < 5:
            rot, ini, fim = span
            exemplos_fn.append({"perdida": f"{rot}:{' '.join(tokens[ini:fim])}",
                                "contexto": " ".join(tokens[max(0, ini - 5):fim + 5])})

print("falsos negativos por rótulo:", dict(fn_por_rotulo.most_common()))
pd.DataFrame(exemplos_fn)

falsos negativos por rótulo: {'LOC': 104, 'MISC': 101, 'ORG': 75, 'PER': 39}


,perdida,contexto
0,LOC:Rancho Alegre d' Oeste,ainda agregava os distridos de Rancho Alegre d' Oeste e Quarto Cen...
1,LOC:Moreira Sales,"então municípios de Mariluz , Moreira Sales e Janiópolis os quais são"
2,MISC:Kodokan,"ensinou seu estilo no centro Kodokan e , eventualmente , estudou"
3,LOC:Estádio Antonio Lins Ribeiro Guimarães,"Seu estádio é o "" Estádio Antonio Lins Ribeiro Guimarães "" com cap..."
4,MISC:Campeonato Mundial de Natação em Piscina Curta de 2010,a seletiva brasileira para o Campeonato Mundial de Natação em Pisc...


## 6. `EntityRuler`: o que o modelo estatístico não cobre

Entidades de formato fixo — CNPJ, CPF, número de lei, código de processo — não precisam de
modelo: uma regra acerta sempre, e o modelo estatístico erraria por não ter visto esses padrões no
treino.

O `EntityRuler` casa **sequências de tokens**, então a primeira coisa a fazer é olhar como o
tokenizador quebra o texto. Regra escrita contra a grafia humana em vez da tokenização real
simplesmente não casa.

In [13]:
texto_br = (
    "A Acme Comércio Ltda, CNPJ 12.345.678/0001-95, foi autuada com base na Lei nº 8.078/1990. "
    "O responsável, José Andrade, CPF 123.456.789-00, recorreu em Brasília."
)

print("como o spaCy quebra esse texto:")
print([t.text for t in nlp.tokenizer(texto_br)])

como o spaCy quebra esse texto:
['A', 'Acme', 'Comércio', 'Ltda', ',', 'CNPJ', '12.345.678/', '0001-95', ',', 'foi', 'autuada', 'com', 'base', 'na', 'Lei', 'nº', '8.078/1990', '.', 'O', 'responsável', ',', 'José', 'Andrade', ',', 'CPF', '123.456.', '789-00', ',', 'recorreu', 'em', 'Brasília', '.']


Note que `12.345.678/0001-95` virou **dois** tokens (`12.345.678/` e `0001-95`), e o CPF
também. Os padrões abaixo são escritos em cima desse fato: cada dicionário da lista descreve um
token, e `REGEX` casa contra o texto daquele token isolado.

In [14]:
padroes = [
    # 12.345.678/ + 0001-95
    {"label": "CNPJ", "pattern": [
        {"TEXT": {"REGEX": r"^\d{2}\.\d{3}\.\d{3}/$"}},
        {"TEXT": {"REGEX": r"^\d{4}-\d{2}$"}},
    ]},
    # 123.456. + 789-00
    {"label": "CPF", "pattern": [
        {"TEXT": {"REGEX": r"^\d{3}\.\d{3}\.$"}},
        {"TEXT": {"REGEX": r"^\d{3}-\d{2}$"}},
    ]},
    # Lei nº 8.078/1990  (o "nº" é opcional)
    {"label": "LEI", "pattern": [
        {"LOWER": {"IN": ["lei", "decreto"]}},
        {"LOWER": {"IN": ["nº", "n.º", "no"]}, "OP": "?"},
        {"TEXT": {"REGEX": r"^[\d\.]+/\d{4}$"}},
    ]},
]

# before="ner": o ruler marca primeiro, e o modelo estatístico respeita o que já está marcado.
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(padroes)

print("pipeline:", nlp.pipe_names)

doc_br = nlp(texto_br)
pd.DataFrame([{"entidade": e.text, "rotulo": e.label_,
               "origem": "regra" if e.label_ in {"CNPJ", "CPF", "LEI"} else "modelo"}
              for e in doc_br.ents])

pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'entity_ruler', 'ner']


,entidade,rotulo,origem
0,Acme Comércio Ltda,LOC,modelo
1,CNPJ,PER,modelo
2,12.345.678/0001-95,CNPJ,regra
3,Lei nº 8.078/1990,LEI,regra
4,José Andrade,PER,modelo
5,CPF,MISC,modelo
6,123.456.789-00,CPF,regra
7,Brasília,LOC,modelo


In [15]:
cores = {"CNPJ": "#ffd54f", "CPF": "#ffd54f", "LEI": "#80cbc4"}
displacy.render(doc_br, style="ent", jupyter=True, options={"colors": cores})

A tabela acima mostra as três regras funcionando — e, de graça, três erros do modelo estatístico no
mesmo texto: `Acme Comércio Ltda` sai como `LOC` em vez de `ORG`, e as próprias siglas `CNPJ` e
`CPF` viram entidades (`PER` e `MISC`). São palavras em maiúscula em posição de nome próprio, num
tipo de documento que não existe na Wikipédia. É a limitação de domínio da seção 7 aparecendo em
três palavras.

In [16]:
# O ruler não interfere na avaliação da seção 4: nenhuma frase da Wikipédia casa esses padrões.
docs_regra = list(nlp.pipe(" ".join(t) for t, _, _ in pares[:500]))
casadas = sum(1 for d in docs_regra for e in d.ents if e.label_ in {"CNPJ", "CPF", "LEI"})
print("entidades de regra encontradas nas 500 primeiras frases do corpus:", casadas)

entidades de regra encontradas nas 500 primeiras frases do corpus: 0


## 7. Limitações e próximos passos

O que os números da seção 4 mostram sobre o `pt_core_news_sm`:

1. **Domínio importa** — o modelo foi treinado em Wikipédia e avaliado em Wikipédia, o que é o
   cenário mais favorável possível. Em texto jurídico, clínico ou em posts de rede social o
   desempenho cai bastante, e a seção 5 já dá o vocabulário para diagnosticar como cai.
2. **`MISC` é um rótulo frouxo** — junta obras, eventos, nacionalidades e produtos. É o pior F1 da
   tabela, e parte disso é desacordo de anotação, não erro do modelo. `ORG` vem em seguida, por
   outro motivo: tem o menor suporte e concorre com `LOC` (times, universidades e empresas levam
   nome de lugar).
3. **Fronteiras seguem a convenção do corpus de treino** — incluir ou não o artigo, o cargo, o
   complemento. Trocar de corpus muda a régua sem que o modelo tenha ficado pior.
4. **Modelo `sm` não tem vetores de palavra** — generaliza pouco para nomes que nunca viu. As
   versões `md` e `lg` têm vetores; `pt_core_news_lg` é o passo natural se a revocação estiver baixa.
5. **Sem resolução de correferência nem canonicalização** — `Petrobras`, `Petróleo Brasileiro S.A.`
   e `a estatal` são três menções da mesma entidade, e o NER não as liga.

### Exercícios

1. Rodar a avaliação com `pt_core_news_lg` (`uv add` do wheel correspondente) e comparar rótulo por
   rótulo com a tabela da seção 4. Onde os vetores ajudam mais?
2. Avaliar em **nível de token** em vez de span exato (comparar as sequências BIO diretamente) e
   explicar por que os números ficam mais altos.
3. Passar `" ".join(tokens)` para `nlp()` em vez de usar `prever_pre_tokenizado` e medir quanto a
   avaliação se degrada por desalinhamento de tokenização.
4. Tratar `MISC` como `O` nos dois lados e recalcular. Quanto do erro vinha só desse rótulo?
5. Escrever padrões de `EntityRuler` para CEP, placa de veículo e número de processo judicial
   (`0000000-00.0000.0.00.0000`), checando antes como o tokenizador quebra cada formato.
6. Treinar um NER próprio: anotar ~100 frases de um domínio seu com o
   [Prodigy](https://prodi.gy) ou à mão em formato `.spacy`, e usar `spacy train` a partir do
   `pt_core_news_sm`. Comparar com o modelo base no mesmo conjunto de teste.